# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring**

I'm choosing this lane because it directly extends the starter pipeline I already ran in Assignment 1. That pipeline showed a hand-written baseline rule achieving Precision@50 ≈ 0.24, while a learned model (random forest) reached Precision@50 ≈ 0.74 — roughly a 3x improvement in picking the right pages to review first. This lane lets me build on that foundation with a proper data contract, a stronger future-looking label, honest validation, and reason codes a real content reviewer could act on. I may revisit this choice by end of Week 4 if clustering (Lane 3) turns out to be a better fit for my interests, but Lane 2 is the strongest and most concrete starting point given what I've already verified works.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which content pages should be prioritized for manual review — for refresh, expansion, or monitoring — given limited reviewer capacity?

**Unit of analysis:** One row = one content page (`content_id`), evaluated using its trailing performance signals (impressions, sessions, position, freshness, etc.) over a defined window.

**The decision this improves:** A content team has limited time and can't manually review every page. Someone has to decide which pages to look at first.

**The action someone takes:** A reviewer opens the top-ranked pages and decides whether to refresh, expand, leave alone, or flag for a larger rewrite — using reason codes (e.g. "stale visible page," "declining with demand," "low CTR visible page") as a starting explanation, not a final verdict.

**Cost of a wrong call:**
- False positive (flagged but didn't need review): wastes limited reviewer time.
- False negative (a genuinely declining page never gets flagged): the site silently loses visibility on a fixable page, possibly for months before anyone notices.
- Because reviewer time is the scarce resource, this is a ranking problem, not a plain yes/no classification — Precision@K matches the real decision better than accuracy.

**Why data/ML can help:** The baseline rule only reached 24% precision in its top 50 picks — 3 out of 4 flagged pages weren't actually worth reviewing first. A model that learns from many signals jointly nearly tripled that precision on the starter data, which is a large enough gap to justify building this out properly.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 anonymized pages) shows:

- **54.2% of pages (16,262 of 30,000) are currently labeled as declining** (`trend_direction == "down"`). This is a large share — over half the dataset — which means there is no shortage of candidate pages for a review/prioritization system to work on. It also means "declining" alone is far too broad to be useful as a review queue; a good scoring system needs to further separate genuine, actionable decline from noise.
- Applying one baseline reason-code rule from the pipeline — "stale but still getting real traffic" (no update in 180+ days, but still ≥500 impressions in the last 90 days) — narrows this down sharply to just **17 pages (0.1% of the dataset)**. This tells me that very few stale pages are still visible enough to matter under this particular threshold, which is a useful early signal: either this specific rule is too strict for this dataset's traffic scale, or genuinely high-value "stale but visible" opportunities are rare and worth prioritizing precisely because they're rare.

Together these numbers support the case for Lane 2: the raw "declining" label is too broad to act on directly (54% of everything), while a single hand-written rule is too narrow (0.1%) — exactly the gap a learned, multi-signal ranking system is meant to close, as the starter pipeline already demonstrated (Precision@50 rising from 0.24 with the baseline rule to 0.74 with a random forest model).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total pages in starter dataset:", len(df))
print("Pages currently declining (trend_direction == 'down'):", (df['trend_direction'] == 'down').sum())
print("Share declining: {:.1%}".format((df['trend_direction'] == 'down').mean()))

stale_visible = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
print("\nPages that are stale but still getting traffic (days_since_last_update >= 180, impressions_90d >= 500):", len(stale_visible))
print("Share of total: {:.1%}".format(len(stale_visible) / len(df)))

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 123 (delta 38), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.84 MiB | 9.12 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/FlyRank-Internship/FlyRank-Internship
Total pages in starter dataset: 30000
Pages currently declining (trend_direction == 'down'): 16262
Share declining: 54.2%

Pages that are stale but still getting traffic (days_since_last_update >= 180, impressions_90d >= 500): 17
Share of total: 0.1%


## 4. Careful words: what I can and can't claim

This is a "look here first" tool, not a "here's the answer" tool. It ranks pages by evidence so a reviewer's limited time goes to the most promising candidates — it doesn't fix anything on its own.

Everything here comes from watching what already happened, not from an experiment, so I can say certain signals tend to show up alongside decline or low engagement — not that they cause it. The "declining" label I'm using is also based on the current 90-day window, not a real future outcome yet, so it's a rough proxy for now, not a validated prediction.

What I won't claim: that refreshing a page will fix its traffic, anything about how Google's algorithm actually works, or anything about AI search rankings — the data only shows whether someone clicked through from an AI tool, not why. And since client names, URLs, and queries were all scrambled before I saw the data, nothing I produce can point back to a real client or page.

Reason codes come with every flagged page so a human reviewer can check my logic and overrule it if they know something the data doesn't.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.